# Step 8 — All Lines, All Levers, Together

Every result so far moved **one lever at a time** on **one line**: the Step 6 gate, Step 7's per-SKU sweep. This step runs the full four-lever factorial across all four lines — the first point interactions become visible, and the evidence base for the **O-10 review**, which follows immediately after this step (D-063).

`src/engine.py` is **broadened in use, not modified** — a test asserts a default `run_scenario` call is bit-identical before and after the sweep.

Grid: 4 values per lever (measured runtime: `run_scenario` ≈ 63ms → 1,024 calls ≈ 65s). Deterministic throughout, no ML, no RNG.

## Setup

In [ ]:
import subprocess, os, sys
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r
REPO = '/content/ibp-tradeoff'
os.chdir('/content'); sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## Rebuild the Step 4 artefacts

In [ ]:
import pandas as pd, numpy as np, yaml
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')
ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

## Upload Step 5a's output

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv:')
uploaded = files.upload()
demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'demand_characteristics: {demand_characteristics.shape} | flagged: {len(flagged)}')

## Build the engine and run the sweep

`sweep_all_lines_all_levers` is the full factorial: 4 lines × 4^4 lever combinations = 1,024 rows. `src/engine.py` is consumed, unchanged.

In [ ]:
from src.engine import TradeOffEngine, LeverSettings, build_line_master
from src.portfolio_sweep import (lever_grid, sweep_all_lines_all_levers,
                                 portfolio_summary, lever_consistency, LEVER_NAMES)

assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))

engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)

grid = lever_grid(assumptions, n=4)
print('grid:')
for k, v in grid.items():
    print(f'  {k:<26}{[round(x,3) for x in v]}')

import time
t0 = time.time()
sweep = sweep_all_lines_all_levers(engine, grid)
print(f'\n{len(sweep)} rows in {time.time()-t0:.0f}s')
sweep.to_csv('portfolio_sweep.csv', index=False)

## Portfolio summary — the O-10 evidence

`worst_case_utilisation_max` and `worst_case_overtime_hours` are read from the **whole grid**, not only from each line's cost-optimal point — a line can be cheap at its optimum but still risk breaching capacity under a plausible alternative setting, and that risk must not be hidden by only looking at the best row.

In [ ]:
summary = portfolio_summary(sweep, assumptions)
summary.to_csv('step08_portfolio_summary.csv', index=False)
print(summary.round(3).to_string(index=False))

## Lever consistency — does L3's story generalise?

For each lever and output, does moving the lever from its lowest to its highest grid value change the output the **same direction** on every line? This is the direct test of whether single-line findings — D-059 (the bias lever says "don't correct" on L3), Step 7's service-target boundary-hugging — hold across the portfolio or were specific to one line.

In [ ]:
consistency = lever_consistency(sweep, grid)
consistency.to_csv('step08_lever_consistency.csv', index=False)
inconsistent = consistency[~consistency.consistent]
print(f'{len(inconsistent)} of {len(consistency)} lever/output pairs are inconsistent across lines:\n')
print(inconsistent.to_string(index=False))

## Tests

In [ ]:
sh('python -m pytest tests/test_portfolio_sweep.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_pipeline.py -q --no-header')

## Consolidated report — the only cell to copy

Always rebuilds rather than reusing anything in memory. Prints module SHAs so the provenance of every number is visible.

In [ ]:
import hashlib, subprocess

t_port = subprocess.run('python -m pytest tests/test_portfolio_sweep.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_eng  = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                        shell=True, capture_output=True, text=True)
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 8 - ALL LINES / ALL LEVERS - CONSOLIDATED REPORT'); w('='*78)
w(f'lines            : {sorted(engine.line_master.line_id.tolist())}')
w(f'grid             : {[len(v) for v in grid.values()]} values/lever, '
  f'{np.prod([len(v) for v in grid.values()])} combos x {len(engine.line_master)} lines')
w(f'assumption set   : {engine.assumption_fingerprint}')
w(f'engine.py sha    : {hashlib.sha256(open("src/engine.py","rb").read()).hexdigest()[:12]}')
w(f'portfolio_sweep sha: {hashlib.sha256(open("src/portfolio_sweep.py","rb").read()).hexdigest()[:12]}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')
w(f'flagged total    : {len(flagged)}')

w(''); w('-- 1. PORTFOLIO SUMMARY '+'-'*54)
w(summary.round(3).to_string(index=False))

w(''); w('-- 2. LEVER CONSISTENCY - inconsistent pairs only '+'-'*27)
w(f'{len(inconsistent)} of {len(consistency)} lever/output pairs inconsistent across lines')
w('')
w(inconsistent.to_string(index=False) if len(inconsistent) else '(all consistent)')

w(''); w('-- 3. CAPACITY - O-10 EVIDENCE '+'-'*47)
breach_lines = summary[summary.any_capacity_breach].line_id.tolist()
w(f'lines with any capacity breach in the grid: {breach_lines or "none"}')
w(summary[['line_id','worst_case_utilisation_max','worst_case_overtime_hours',
          'any_capacity_breach']].round(3).to_string(index=False))

w(''); w('-- 4. TESTS '+'-'*66)
for label, r in (('test_portfolio_sweep.py', t_port), ('test_engine.py', t_eng),
                 ('test_pipeline.py', t_pipe)):
    w(f'{label:<24}: ' + (r.stdout.strip().splitlines() or ['no output'])[-1])
if any(r.returncode for r in (t_port, t_eng, t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_port, t_eng, t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 5. CHECKS '+'-'*65)
checks = [
 ('every line swept', set(sweep.line_id) == set(engine.line_master.line_id)),
 ('capacity shortfall always finite and non-negative',
  bool(np.isfinite(sweep.capacity_shortfall_total).all()
       and (sweep.capacity_shortfall_total >= 0).all())),
 ('at least one interior optimum found somewhere in the portfolio',
  bool(summary.filter(like='interior_optimum_').any().any())),
 ('all test suites pass', all(r.returncode==0 for r in (t_port,t_eng,t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('This step reports evidence for the O-10 review. It does not conclude O-10 -')
w('that judgement belongs to the review itself, using this output.')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step08_report.txt','w').write(report_text)
try:
    import shutil
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step08_report.txt','portfolio_sweep.csv',
                  'step08_portfolio_summary.csv','step08_lever_consistency.csv'):
            if os.path.exists(f): shutil.copy(f, d)
        print('saved to', d, '\\n')
except Exception as e:
    print('Drive copy skipped:', e, '\\n')
print(report_text)